# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM057_Household_composition_by_age |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |RM057_Household_composition_by_age|
| Output File | |
| Notebook Author | Peter |
| Last Updated | |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json
from pathlib import Path

In [2]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [3]:
# File paths

input_file = "RM057_Household_composition_by_age.xlsx"
output_file = "household_by_age.json"

raw_data_path = Path("../../data/demographic/raw/household_by_age/")
output_data_path = Path("../../data/demographic/processed/household_by_age/")

## 3. Load Dataset

In [4]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Household composition (15 categories) Code,Household composition (15 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0,Aged under 1 year,0
1,E06000001,Hartlepool,-8,Does not apply,1,Aged 1 year,0
2,E06000001,Hartlepool,-8,Does not apply,2,Aged 2 years,0
3,E06000001,Hartlepool,-8,Does not apply,3,Aged 3 years,0
4,E06000001,Hartlepool,-8,Does not apply,4,Aged 4 years,0


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [5]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425700 entries, 0 to 425699
Data columns (total 7 columns):
 #   Column                                      Non-Null Count   Dtype 
---  ------                                      --------------   ----- 
 0   Lower tier local authorities Code           425700 non-null  object
 1   Lower tier local authorities                425700 non-null  object
 2   Household composition (15 categories) Code  425700 non-null  int64 
 3   Household composition (15 categories)       425700 non-null  object
 4   Age (86 categories) Code                    425700 non-null  int64 
 5   Age (86 categories)                         425700 non-null  object
 6   Observation                                 425700 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 22.7+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Household composition (15 categories) Code',
       'Household composition (15 categories)', 'Age (86 categories) Code',
       'Age (86 categories)', 'Observation'],
      dtype='object')

In [6]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Household composition (15 categories) Code,Household composition (15 categories),Age (86 categories) Code,Age (86 categories),Observation
count,425700,425700,425700.000000,425700,425700.000000,425700,425700.000000
unique,330,330,NaN,15,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Aged under 1 year,NaN
freq,1290,1290,NaN,28380,NaN,4950,NaN
mean,NaN,NaN,6.466667,NaN,42.500000,NaN,137.547461
std,NaN,NaN,5.487771,NaN,24.824412,NaN,297.901125
min,NaN,NaN,-8.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,3.000000,NaN,21.000000,NaN,0.000000
50%,NaN,NaN,7.000000,NaN,42.500000,NaN,29.000000
75%,NaN,NaN,11.000000,NaN,64.000000,NaN,143.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [2]:
# Cleaning code

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [9]:
# Dataset-specific processing
def age_group(age):
    if age <= 4:
        return "0-4"
    elif age <= 11:
        return "5-11"
    elif age <= 17:
        return "12-17"
    elif age <= 22:
        return "18-22"
    elif age <= 34:
        return "23-34"
    elif age <= 49:
        return "35-49"
    elif age <= 64:
        return "50-64"
    elif age <= 74:
        return "65-74"
    else:
        return "75+"
    
age_order = [ "0-4",
             "5-11",
             "12-17",
             "18-22",
             "23-34",
             "35-49",
             "50-64",
             "65-74",
             "75+"]

In [7]:
# Dataset-specific processing
household_by_age_ontology = {

    # One-person households
    "One-person household": [
        "One-person household: Aged 66 years and over",
        "One-person household: Other"
    ],

    # Couple households
    "Couple household: No children": [
        "Single family household: Married or civil partnership couple: No children",
        "Single family household: Cohabiting couple family: No children"
    ],

    "Couple household: With dependent children": [
        "Single family household: Married or civil partnership couple: Dependent children",
        "Single family household: Cohabiting couple family: With dependent children"
    ],

    "Couple household: All children non-dependent": [
        "Single family household: Married or civil partnership couple: All children non-dependent",
        "Single family household: Cohabiting couple family: All children non-dependent"
    ],

    # Lone parent households
    "Lone parent household: With dependent children": [
        "Single family household: Lone parent family: With dependent children"
    ],

    "Lone parent household: All children non-dependent": [
        "Single family household: Lone parent family: All children non-dependent"
    ],

    # Older family households
    "Older family household": [
        "Single family household: All aged 66 years and over"
    ],

    # Other family households
    "Other family household": [
        "Single family household: Other single family household: Other family composition",
        "Other household types: Other related household: Other family composition"
    ],

    # Other non-family households
    "Other non-family household": [
        "Other household types: With dependent children",
        "Other household types: Other, including all full-time students and all aged 66 years and over"
    ]
}

In [8]:
selected_columns = ['Lower tier local authorities', 'Household composition (15 categories)',
                                     'Age (86 categories) Code', 'Observation']
df_processed = df_processed[selected_columns]

In [10]:
def get_household_by_age_distribution(local_authority):
    filtered_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    filtered_data['Age Group'] = filtered_data['Age (86 categories) Code'].apply(age_group)

    result = (filtered_data.groupby(['Lower tier local authorities', 'Household composition (15 categories)', 'Age Group'], as_index=False)['Observation'].sum())

    result['Age Group'] = pd.Categorical(result['Age Group'], categories=age_order, ordered=True)

    result = result.reset_index(drop=True)

    return result

In [11]:
get_household_by_age_distribution("Manchester")

,Lower tier local authorities,Household composition (15 categories),Age Group,Observation
0,Manchester,Does not apply,0-4,0
1,Manchester,Does not apply,12-17,0
2,Manchester,Does not apply,18-22,0
3,Manchester,Does not apply,23-34,0
4,Manchester,Does not apply,35-49,0
...,...,...,...,...
130,Manchester,Single family household: Married or civil part...,35-49,6300
131,Manchester,Single family household: Married or civil part...,5-11,0
132,Manchester,Single family household: Married or civil part...,50-64,11011
133,Manchester,Single family household: Married or civil part...,65-74,2721


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [12]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425700 entries, 0 to 425699
Data columns (total 4 columns):
 #   Column                                 Non-Null Count   Dtype 
---  ------                                 --------------   ----- 
 0   Lower tier local authorities           425700 non-null  object
 1   Household composition (15 categories)  425700 non-null  object
 2   Age (86 categories) Code               425700 non-null  int64 
 3   Observation                            425700 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 13.0+ MB


Lower tier local authorities             0
Household composition (15 categories)    0
Age (86 categories) Code                 0
Observation                              0
dtype: int64

## 8. Export Processed Dataset

In [13]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

household_by_age_df = get_household_by_age_distribution("Manchester")

household_by_age_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported